# 🔢 Colab 2 — Embeddings Semánticos vs TF-IDF
### Materia: Procesamiento de Lenguaje Natural — Nivel Terciario

---

## El problema central de este colab

Ya conocen TF-IDF: representa un documento como un vector donde cada dimensión es el peso de un término. Es potente para muchas tareas, pero tiene una limitación fundamental:

> **TF-IDF busca coincidencia de tokens. No entiende significado.**

Si el documento dice *"rescisión del contrato"* y el usuario pregunta *"¿me pueden echar?"*, TF-IDF no encuentra ninguna relación porque no comparten palabras.

Los **embeddings semánticos** resuelven exactamente esto: mapean el texto a un espacio vectorial donde textos con significado similar quedan cerca, aunque usen palabras completamente distintas.

## Lo que vamos a comparar

| Sistema | Cómo representa el texto | Qué puede encontrar |
|---|---|---|
| **TF-IDF** | Frecuencia de términos (sparse) | Coincidencia de palabras |
| **Embedding semántico** | Vector denso aprendido (dense) | Similitud de significado |

> 🎯 **Objetivo:** ver empíricamente cuándo TF-IDF falla y los embeddings triunfan, y viceversa.

In [1]:
!pip install chromadb sentence-transformers openai scikit-learn --quiet
print('✅ Dependencias instaladas')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sou

In [2]:
import os
import numpy as np
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.getenv('OPENAI_API_KEY', 'TU_API_KEY_ACA')

client = OpenAI(api_key=api_key)
print('✅ Cliente OpenAI inicializado')

✅ Cliente OpenAI inicializado


---
## 📄 El corpus

Usamos el mismo reglamento de TechStore del Colab anterior, dividido en chunks por párrafo.
Esto nos permite comparar los dos sistemas de retrieval en igualdad de condiciones.

In [3]:
CHUNKS = [
    "Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos contados desde la fecha de entrega. Para productos con defectos de fábrica, el plazo se extiende a 90 días. Las devoluciones fuera de estos plazos no serán aceptadas salvo orden judicial.",
    "El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ticket o factura de compra. Los productos de higiene personal, auriculares in-ear y colchones no admiten devolución por razones sanitarias.",
    "El cliente debe iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta. Una vez aprobada la solicitud, recibirá un código de autorización RMA. Las devoluciones sin código RMA serán rechazadas en el depósito. El reembolso se acredita dentro de los 10 días hábiles.",
    "Todos los productos tienen garantía legal mínima de 6 meses por defectos de fabricación, conforme a la Ley 24.240 de Defensa del Consumidor. Esta garantía es obligatoria y no puede ser reducida por ningún acuerdo contractual.",
    "Los televisores de 40 pulgadas o más tienen garantía de 12 meses. Las notebooks, tablets y computadoras tienen garantía de 12 meses. Los smartphones de gama alta tienen garantía de 12 meses. Los accesorios y cables tienen garantía de 3 meses.",
    "La garantía no cubre daños causados por mal uso, golpes, líquidos, modificaciones no autorizadas o uso fuera de las condiciones especificadas por el fabricante. Los daños por sobretensión solo están cubiertos con estabilizador de tensión certificado.",
    "Para hacer efectiva la garantía el cliente debe presentar el producto en cualquier sucursal. El tiempo de reparación estimado es de 15 días hábiles. Si no puede completarse, TechStore ofrecerá un equipo de reemplazo o el reembolso del precio de compra.",
    "TechStore Argentina realiza envíos a todo el territorio nacional. Las zonas remotas son aquellas sin servicio de correo regular o a más de 50 km del centro de distribución más cercano y pueden tener plazos y costos adicionales.",
    "Para CABA y GBA el plazo de entrega es de 24 a 48 horas hábiles. Para el interior del país es de 3 a 7 días hábiles. Patagonia y NOA tienen un plazo de 7 a 12 días hábiles. Los plazos son estimativos y pueden verse afectados por feriados o fuerza mayor.",
    "El envío es gratuito para compras iguales o superiores a $50.000 en todo el país. Para compras menores el costo se calcula según el peso y la distancia. Los productos de más de 30 kg tienen un cargo adicional de logística pesada.",
    "Una vez despachado el pedido el cliente recibirá un número de seguimiento por email y SMS. Si no hay nadie al momento de la entrega el transportista dejará un aviso y realizará un segundo intento al día siguiente. Después de dos intentos fallidos el paquete vuelve al depósito.",
    "Se aceptan tarjetas Visa, Mastercard, American Express y Cabal. Las tarjetas prepagas solo se aceptan para compras de hasta $30.000. También se aceptan tarjetas de débito de las mismas redes.",
    "Con tarjetas Visa y Mastercard se ofrecen 3, 6 y 12 cuotas sin interés en productos seleccionados. Los productos en liquidación no participan del programa de cuotas sin interés. Las cuotas no son acumulables con otros descuentos.",
    "Los pagos por transferencia bancaria reciben un descuento adicional del 10% sobre el precio de lista. La transferencia debe acreditarse dentro de las 48 horas de realizada la compra, caso contrario la orden se cancela automáticamente.",
    "Se aceptan pagos con Mercado Pago, Modo, Ualá y Naranja X con las mismas condiciones que las tarjetas de débito. Pueden existir promociones específicas con alguna billetera digital comunicadas en el sitio web.",
    "El centro de atención al cliente opera de lunes a viernes de 9 a 18 hs y los sábados de 9 a 13 hs. Los canales disponibles son WhatsApp, email a soporte@techstore.com.ar, chat en vivo y atención presencial en sucursales.",
    "Los mensajes de WhatsApp y chat tienen respuesta máxima de 2 horas durante el horario de atención. Los emails se responden dentro de las 24 horas hábiles. Los reclamos formales se resuelven en un máximo de 72 horas hábiles.",
    "Si el cliente no está satisfecho con la resolución puede solicitar escalamiento al área de Supervisión. Si persiste la insatisfacción puede presentar el reclamo ante la Dirección de Defensa del Consumidor de su provincia."
]

print(f"📄 Corpus listo: {len(CHUNKS)} chunks")
for i, chunk in enumerate(CHUNKS):
    print(f"  [{i:02d}] {chunk[:70]}...")

📄 Corpus listo: 18 chunks
  [00] Los clientes tienen derecho a solicitar la devolución de productos den...
  [01] El producto debe presentarse en su embalaje original, sin señales de u...
  [02] El cliente debe iniciar el proceso de devolución a través del portal w...
  [03] Todos los productos tienen garantía legal mínima de 6 meses por defect...
  [04] Los televisores de 40 pulgadas o más tienen garantía de 12 meses. Las ...
  [05] La garantía no cubre daños causados por mal uso, golpes, líquidos, mod...
  [06] Para hacer efectiva la garantía el cliente debe presentar el producto ...
  [07] TechStore Argentina realiza envíos a todo el territorio nacional. Las ...
  [08] Para CABA y GBA el plazo de entrega es de 24 a 48 horas hábiles. Para ...
  [09] El envío es gratuito para compras iguales o superiores a $50.000 en to...
  [10] Una vez despachado el pedido el cliente recibirá un número de seguimie...
  [11] Se aceptan tarjetas Visa, Mastercard, American Express y Cabal. Las ta...
  

---
# 🔬 Sistema 1 — TF-IDF

Construimos el retriever con TF-IDF clásico usando scikit-learn, tal como lo conocen de la materia.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Construimos el índice TF-IDF
vectorizer = TfidfVectorizer()
matriz_tfidf = vectorizer.fit_transform(CHUNKS)

print(f"✅ Índice TF-IDF construido")
print(f"   Documentos: {matriz_tfidf.shape[0]}")
print(f"   Vocabulario: {matriz_tfidf.shape[1]} términos únicos")
print(f"   Tipo de matriz: sparse (la mayoría son ceros)")

def recuperar_tfidf(pregunta, top_k=3):
    """Recupera los chunks más relevantes usando TF-IDF + cosine similarity."""
    vec_pregunta = vectorizer.transform([pregunta])
    similitudes  = cosine_similarity(vec_pregunta, matriz_tfidf)[0]
    indices_top  = similitudes.argsort()[::-1][:top_k]
    return [(CHUNKS[i], float(similitudes[i])) for i in indices_top]

# Test rápido
resultados = recuperar_tfidf("devolución de productos", top_k=2)
print(f"\n🔍 Test — 'devolución de productos':")
for chunk, score in resultados:
    print(f"  Score: {score:.4f} | {chunk[:80]}...")

✅ Índice TF-IDF construido
   Documentos: 18
   Vocabulario: 323 términos únicos
   Tipo de matriz: sparse (la mayoría son ceros)

🔍 Test — 'devolución de productos':
  Score: 0.3272 | Los clientes tienen derecho a solicitar la devolución de productos dentro de los...
  Score: 0.2398 | El producto debe presentarse en su embalaje original, sin señales de uso, con to...


---
# 🔬 Sistema 2 — Embeddings Semánticos

Ahora construimos el retriever con embeddings de OpenAI. A diferencia de TF-IDF, cada chunk se convierte en un vector denso de 1536 dimensiones que captura su significado.

In [5]:
import chromadb
from chromadb.utils import embedding_functions

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("colab2_embeddings")
except:
    pass

coleccion = chroma_client.create_collection(
    name="colab2_embeddings",
    embedding_function=openai_ef
)

print("⏳ Generando embeddings e indexando...")
coleccion.add(
    documents=CHUNKS,
    ids=[f"chunk_{i}" for i in range(len(CHUNKS))]
)
print(f"✅ Vector store creado")
print(f"   Documentos indexados: {coleccion.count()}")
print(f"   Modelo: text-embedding-3-small (1536 dimensiones, vector denso)")

def recuperar_embedding(pregunta, top_k=3):
    """Recupera los chunks más relevantes usando embeddings semánticos."""
    resultados = coleccion.query(query_texts=[pregunta], n_results=top_k)
    return list(zip(resultados['documents'][0], resultados['distances'][0]))

⏳ Generando embeddings e indexando...
✅ Vector store creado
   Documentos indexados: 18
   Modelo: text-embedding-3-small (1536 dimensiones, vector denso)


---
# 🧪 Experimento 1 — Preguntas léxicas (TF-IDF debería funcionar bien)

Primero probamos con preguntas que usan las mismas palabras que el corpus.
En este caso TF-IDF tiene ventaja porque hay coincidencia directa de términos.

In [6]:
def comparar(pregunta, top_k=2):
    print(f"\n❓ Pregunta: '{pregunta}'")
    print("=" * 70)

    print("\n📊 TF-IDF:")
    print("-" * 40)
    for chunk, score in recuperar_tfidf(pregunta, top_k):
        indicador = "✅" if score > 0.05 else "⚠️ "
        print(f"{indicador} Score: {score:.4f} | {chunk[:120]}...")

    print("\n🔢 Embedding semántico:")
    print("-" * 40)
    for chunk, dist in recuperar_embedding(pregunta, top_k):
        indicador = "✅" if dist < 0.5 else "⚠️ "
        print(f"{indicador} Dist:  {dist:.4f} | {chunk[:120]}...")

# Preguntas que usan palabras del corpus directamente
comparar("devolución embalaje original accesorios")
comparar("garantía notebooks tablets 12 meses")


❓ Pregunta: 'devolución embalaje original accesorios'

📊 TF-IDF:
----------------------------------------
✅ Score: 0.3531 | El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ti...
✅ Score: 0.0600 | Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos contados desde la fech...

🔢 Embedding semántico:
----------------------------------------
⚠️  Dist:  0.7033 | El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ti...
⚠️  Dist:  1.0190 | El cliente debe iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta. Una vez aprobada la so...

❓ Pregunta: 'garantía notebooks tablets 12 meses'

📊 TF-IDF:
----------------------------------------
✅ Score: 0.6514 | Los televisores de 40 pulgadas o más tienen garantía de 12 meses. Las notebooks, tablets y computadoras tienen garantía ...
✅ Score: 0.

### 🔍 ¿Qué observás?

Con preguntas que usan los mismos términos del corpus, TF-IDF funciona razonablemente bien. Los scores son altos donde hay coincidencia de palabras clave.

Ahora viene la parte interesante 👇

---
# 🧪 Experimento 2 — El momento "ahá": preguntas semánticas

Ahora hacemos preguntas usando **vocabulario completamente distinto** al del corpus. El usuario pregunta con sus propias palabras, no con las del reglamento.

> 💡 Esto es exactamente lo que pasa en producción: los usuarios no leen el reglamento antes de preguntar.

In [8]:
# Pares: (pregunta del usuario, lo que dice el corpus)
preguntas_semanticas = [
    (
        "¿me pueden echar del contrato si no pago?",
        # El corpus dice: 'la orden será cancelada automáticamente'
    ),
    (
        "¿puedo arrepentirme de una compra?",
        # El corpus dice: 'devolución dentro de los 30 días'
    ),
    (
        "el producto se rompió solo, ¿qué hago?",
        # El corpus dice: 'garantía por defectos de fabricación'
    ),
    (
        "¿cuánto tarda en llegar si vivo lejos de Buenos Aires?",
        # El corpus dice: 'interior del país 3 a 7 días hábiles'
    ),
    (
        "no tengo tarjeta de crédito, ¿puedo igual comprar?",
        # El corpus dice: 'transferencia bancaria', 'billeteras digitales'
    ),
]

for pregunta_tuple in preguntas_semanticas:
    # Extract only the question string from the tuple
    pregunta = pregunta_tuple[0]
    comparar(pregunta, top_k=1)
    print()


❓ Pregunta: '¿me pueden echar del contrato si no pago?'

📊 TF-IDF:
----------------------------------------
✅ Score: 0.2101 | Si el cliente no está satisfecho con la resolución puede solicitar escalamiento al área de Supervisión. Si persiste la i...

🔢 Embedding semántico:
----------------------------------------
⚠️  Dist:  1.1991 | Si el cliente no está satisfecho con la resolución puede solicitar escalamiento al área de Supervisión. Si persiste la i...


❓ Pregunta: '¿puedo arrepentirme de una compra?'

📊 TF-IDF:
----------------------------------------
✅ Score: 0.1600 | Para hacer efectiva la garantía el cliente debe presentar el producto en cualquier sucursal. El tiempo de reparación est...

🔢 Embedding semántico:
----------------------------------------
⚠️  Dist:  1.0658 | Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos contados desde la fech...


❓ Pregunta: 'el producto se rompió solo, ¿qué hago?'

📊 TF-IDF:
--------------------

### 🔍 Análisis del experimento

Observá en cada caso:

**TF-IDF:** probablemente devuelve score 0.0 o muy bajo en las preguntas semánticas porque no hay palabras en común. *"Arrepentirme"* no está en el corpus; *"devolución"* sí.

**Embeddings:** entiende que *"arrepentirme de una compra"* y *"devolución"* son conceptos relacionados, y encuentra el chunk correcto aunque no compartan ni una palabra.

> ⚠️ **Esto no significa que TF-IDF sea inútil.** Para búsqueda de términos exactos (nombres propios, códigos, números de artículo) TF-IDF es más preciso. La clave es entender cuándo usar cada uno.

---
# 🧪 Experimento 3 — Sparse vs Dense: visualizando la diferencia

Una forma de entender la diferencia es ver cómo representa cada sistema el mismo texto.

In [9]:
# Veamos cómo representa TF-IDF un chunk
chunk_ejemplo = CHUNKS[4]  # el de garantías por categoría
print("📄 Chunk de ejemplo:")
print(chunk_ejemplo)
print()

# Representación TF-IDF
vec_tfidf = vectorizer.transform([chunk_ejemplo])
terminos  = vectorizer.get_feature_names_out()
scores    = vec_tfidf.toarray()[0]
top10_idx = scores.argsort()[::-1][:10]

print("📊 TF-IDF — Top 10 términos con mayor peso:")
print(f"   Dimensiones totales del vector: {len(scores)}")
print(f"   Dimensiones no-cero: {(scores > 0).sum()} (el resto son 0)")
print()
for idx in top10_idx:
    if scores[idx] > 0:
        print(f"   '{terminos[idx]}': {scores[idx]:.4f}")

📄 Chunk de ejemplo:
Los televisores de 40 pulgadas o más tienen garantía de 12 meses. Las notebooks, tablets y computadoras tienen garantía de 12 meses. Los smartphones de gama alta tienen garantía de 12 meses. Los accesorios y cables tienen garantía de 3 meses.

📊 TF-IDF — Top 10 términos con mayor peso:
   Dimensiones totales del vector: 323
   Dimensiones no-cero: 19 (el resto son 0)

   'meses': 0.5014
   'garantía': 0.4114
   'tienen': 0.3521
   '12': 0.3380
   'de': 0.2643
   'los': 0.1823
   'pulgadas': 0.1432
   'computadoras': 0.1432
   'notebooks': 0.1432
   'cables': 0.1432


In [10]:
# Ahora veamos el embedding
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunk_ejemplo
)
embedding = response.data[0].embedding

print(f"🔢 Embedding semántico:")
print(f"   Dimensiones: {len(embedding)}")
print(f"   Todos los valores son distintos de cero: {all(v != 0 for v in embedding)}")
print(f"   Primeras 10 dimensiones: {[round(v, 4) for v in embedding[:10]]}")
print()
print("💡 A diferencia de TF-IDF:")
print("   - No hay términos con nombres, son dimensiones abstractas")
print("   - Todas las dimensiones tienen valores (vector denso)")
print("   - El significado está distribuido en TODO el vector")
print("   - Dos textos similares tendrán vectores similares aunque no compartan palabras")

🔢 Embedding semántico:
   Dimensiones: 1536
   Todos los valores son distintos de cero: True
   Primeras 10 dimensiones: [-0.0103, 0.0381, 0.0475, 0.0068, -0.0344, -0.0127, -0.0025, -0.0161, -0.0076, -0.012]

💡 A diferencia de TF-IDF:
   - No hay términos con nombres, son dimensiones abstractas
   - Todas las dimensiones tienen valores (vector denso)
   - El significado está distribuido en TODO el vector
   - Dos textos similares tendrán vectores similares aunque no compartan palabras


---
# 🧪 Experimento 4 — Similitud semántica directa

Podemos usar los embeddings para medir similitud entre cualquier par de textos, sin necesidad de un corpus.

In [11]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import numpy as np

def similitud_semantica(texto_a, texto_b):
    """Calcula similitud coseno entre dos textos usando embeddings."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[texto_a, texto_b]
    )
    vec_a = np.array(response.data[0].embedding).reshape(1, -1)
    vec_b = np.array(response.data[1].embedding).reshape(1, -1)
    return float(cos_sim(vec_a, vec_b)[0][0])

def similitud_tfidf(texto_a, texto_b):
    """Calcula similitud coseno entre dos textos usando TF-IDF."""
    vecs = vectorizer.transform([texto_a, texto_b])
    return float(cos_sim(vecs[0], vecs[1])[0][0])

# Pares a comparar: (texto_a, texto_b, relación esperada)
pares = [
    ("devolución de productos",     "arrepentimiento de compra",       "semántica"),
    ("garantía por defecto",        "el producto se rompió solo",       "semántica"),
    ("envío al interior del país",  "despacho a provincias argentinas", "semántica"),
    ("Mercado Pago",                "billetera digital",                "semántica"),
    ("devolución de productos",     "garantía de productos",            "relacionada"),
    ("política de envíos",          "medios de pago",                   "no relacionada"),
    ("devolución de productos",     "devolución de compras online",     "casi igual"),
]

print(f"{'Par':<55} {'TF-IDF':>8} {'Embedding':>10} {'Tipo'}")
print("=" * 85)
for a, b, tipo in pares:
    s_tfidf = similitud_tfidf(a, b)
    s_emb   = similitud_semantica(a, b)
    print(f"'{a[:22]}' ↔ '{b[:22]}'  {s_tfidf:>8.4f}  {s_emb:>10.4f}  {tipo}")

Par                                                       TF-IDF  Embedding Tipo
'devolución de producto' ↔ 'arrepentimiento de com'    0.1043      0.5362  semántica
'garantía por defecto' ↔ 'el producto se rompió '    0.0000      0.3973  semántica
'envío al interior del ' ↔ 'despacho a provincias '    0.0000      0.5239  semántica
'Mercado Pago' ↔ 'billetera digital'    0.0000      0.4352  semántica
'devolución de producto' ↔ 'garantía de productos'    0.4849      0.6731  relacionada
'política de envíos' ↔ 'medios de pago'    0.0864      0.4232  no relacionada
'devolución de producto' ↔ 'devolución de compras '    0.5466      0.7393  casi igual


### 🔍 Lo que debería mostrar esta tabla

Para pares **semánticos** (misma idea, palabras distintas):
- TF-IDF → similitud cercana a 0 (no comparten palabras)
- Embedding → similitud alta (entiende la relación)

Para pares **no relacionados**:
- Ambos deberían dar similitud baja

Para pares **casi iguales**:
- Ambos deberían dar similitud alta (aquí TF-IDF también funciona bien)

> 💡 Esto ilustra por qué los embeddings revolucionaron la búsqueda: permiten encontrar información relevante aunque el usuario no use las palabras exactas del documento.

---
# 🧪 Experimento 5 — Búsqueda híbrida (bonus)

En producción muchas veces se combina TF-IDF con embeddings: **Hybrid Search**.

La idea es simple: combinar el score de TF-IDF (bueno para términos exactos) con el score de similitud semántica (bueno para conceptos). Esto da lo mejor de los dos mundos.

In [12]:
def recuperar_hibrido(pregunta, top_k=3, peso_tfidf=0.3, peso_embedding=0.7):
    """
    Combina TF-IDF y embeddings para retrieval híbrido.
    peso_tfidf + peso_embedding deben sumar 1.
    """
    # Scores TF-IDF (normalizados entre 0 y 1)
    scores_tfidf_raw = [score for _, score in recuperar_tfidf(pregunta, top_k=len(CHUNKS))]
    max_tfidf = max(scores_tfidf_raw) if max(scores_tfidf_raw) > 0 else 1
    scores_tfidf = [s / max_tfidf for s in scores_tfidf_raw]

    # Scores embedding (distancia → similitud: 1 - distancia)
    resultados_emb = coleccion.query(query_texts=[pregunta], n_results=len(CHUNKS))
    distancias     = resultados_emb['distances'][0]
    scores_emb     = [1 - d for d in distancias]

    # Los resultados de embedding pueden no estar en el mismo orden que los chunks
    ids_emb    = [int(id_.split('_')[1]) for id_ in resultados_emb['ids'][0]]
    emb_por_id = dict(zip(ids_emb, scores_emb))

    # Combinamos
    scores_combinados = []
    for i in range(len(CHUNKS)):
        score_hibrido = peso_tfidf * scores_tfidf[i] + peso_embedding * emb_por_id.get(i, 0)
        scores_combinados.append((i, score_hibrido))

    top = sorted(scores_combinados, key=lambda x: x[1], reverse=True)[:top_k]
    return [(CHUNKS[i], score) for i, score in top]

# Comparamos los tres sistemas en una pregunta semántica
pregunta = "¿puedo arrepentirme de una compra online?"
print(f"❓ Pregunta: '{pregunta}'")
print("=" * 70)

print("\n📊 Solo TF-IDF:")
for chunk, score in recuperar_tfidf(pregunta, top_k=2):
    print(f"  Score: {score:.4f} | {chunk[:100]}...")

print("\n🔢 Solo Embedding:")
for chunk, dist in recuperar_embedding(pregunta, top_k=2):
    print(f"  Dist:  {dist:.4f} | {chunk[:100]}...")

print("\n⚡ Híbrido (30% TF-IDF + 70% Embedding):")
for chunk, score in recuperar_hibrido(pregunta, top_k=2):
    print(f"  Score: {score:.4f} | {chunk[:100]}...")

❓ Pregunta: '¿puedo arrepentirme de una compra online?'

📊 Solo TF-IDF:
  Score: 0.1600 | Para hacer efectiva la garantía el cliente debe presentar el producto en cualquier sucursal. El tiem...
  Score: 0.1432 | Los pagos por transferencia bancaria reciben un descuento adicional del 10% sobre el precio de lista...

🔢 Solo Embedding:
  Dist:  1.0401 | El cliente debe iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta. Un...
  Dist:  1.0509 | Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos co...

⚡ Híbrido (30% TF-IDF + 70% Embedding):
  Score: 0.2643 | Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos co...
  Score: 0.2390 | El cliente debe iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta. Un...


---
# 🧪 Zona de experimentación libre

In [13]:
# ✏️ Probá tus propias preguntas y observá las diferencias

MIS_PREGUNTAS = [
    "¿qué hago si me llegó algo roto?",
    "quiero cancelar mi pedido",
    "cuotas sin cargo adicional",
    # Agregá las tuyas acá
]

for pregunta in MIS_PREGUNTAS:
    print(f"\n❓ '{pregunta}'")
    print("-" * 50)

    res_tfidf = recuperar_tfidf(pregunta, top_k=1)
    res_emb   = recuperar_embedding(pregunta, top_k=1)

    print(f"  TF-IDF    (score {res_tfidf[0][1]:.4f}): {res_tfidf[0][0][:100]}...")
    print(f"  Embedding (dist  {res_emb[0][1]:.4f}): {res_emb[0][0][:100]}...")


❓ '¿qué hago si me llegó algo roto?'
--------------------------------------------------
  TF-IDF    (score 0.3056): Si el cliente no está satisfecho con la resolución puede solicitar escalamiento al área de Supervisi...
  Embedding (dist  1.1315): El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios i...

❓ 'quiero cancelar mi pedido'
--------------------------------------------------
  TF-IDF    (score 0.1160): El cliente debe iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta. Un...
  Embedding (dist  1.1464): Los pagos por transferencia bancaria reciben un descuento adicional del 10% sobre el precio de lista...

❓ 'cuotas sin cargo adicional'
--------------------------------------------------
  TF-IDF    (score 0.3874): Con tarjetas Visa y Mastercard se ofrecen 3, 6 y 12 cuotas sin interés en productos seleccionados. L...
  Embedding (dist  1.0217): Con tarjetas Visa y Mastercard se ofrecen 3, 6 y 12 cuotas

In [14]:
# 🔍 Para cualquier pregunta: ver el vector TF-IDF completo
MI_PREGUNTA = "me arrepentí de la compra"

vec = vectorizer.transform([MI_PREGUNTA]).toarray()[0]
terminos_no_cero = [(vectorizer.get_feature_names_out()[i], vec[i])
                    for i in range(len(vec)) if vec[i] > 0]

print(f"Vector TF-IDF de: '{MI_PREGUNTA}'")
if terminos_no_cero:
    print("Términos con peso > 0:")
    for termino, peso in sorted(terminos_no_cero, key=lambda x: -x[1]):
        print(f"  '{termino}': {peso:.4f}")
else:
    print("⚠️  Ningún término de esta pregunta aparece en el vocabulario del corpus.")
    print("   Esto significa que TF-IDF no puede recuperar nada relevante.")
    print("   Score será 0 para todos los documentos.")

Vector TF-IDF de: 'me arrepentí de la compra'
Términos con peso > 0:
  'compra': 0.7994
  'la': 0.5131
  'de': 0.3125


---
# ✅ Resumen del Colab 2

| Característica | TF-IDF | Embedding semántico |
|---|---|---|
| **Representación** | Vector sparse (muchos ceros) | Vector denso (todas las dims) |
| **Dimensiones** | = tamaño del vocabulario | Fijo (ej: 1536) |
| **Entiende sinónimos** | ❌ No | ✅ Sí |
| **Términos exactos** | ✅ Muy preciso | ⚠️ Puede fallar |
| **Costo de indexación** | Muy bajo | Requiere llamada a API |
| **Mejor para** | Búsqueda de keywords, códigos | Búsqueda semántica, QA |

## 🔑 Conclusión clave

No hay un ganador absoluto. En sistemas de producción se suele usar **búsqueda híbrida**: TF-IDF para términos exactos (números de orden, nombres propios, códigos de producto) + embeddings para intención semántica.

## 🚀 Siguiente: Colab 3 — Prompt Engineering
Mismo retrieval, distintos prompts. ¿Cuánto importa cómo le pedís al LLM que use el contexto?